In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

In [2]:
DATA_PATH = "../data/processed/merged_dataset.parquet"

HAZARD_MODEL_PATH = "../data/exports/hazard_model.pkl"

DISTANCE_MODEL_PATH = "../data/exports/distance_model.pkl"

VELOCITY_MODEL_PATH = "../data/exports/velocity_model.pkl"

OUTPUT_JSON = "../data/exports/asteroid_trajectories.json"

TOP_N_ASTEROIDS = 50

N_TRAJECTORY_POINTS = 720

In [3]:
df = pd.read_parquet(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Rows: 27422
Columns: 44


,designation,full_name,close_approach_date,distance_au,dist_km,dist_lunar,distance_min_au,distance_max_au,velocity_km_s,v_rel_kmh,...,first_obs,last_obs,data_arc,data_arc_years,orbit_stretch,interaction_index,size_energy_index,diameter_velocity_proxy,year,month
0,2022 AP1,(2022 AP1),2015-01-01 00:27:00,0.045100,6746806.0,17.55,0.013778,0.076571,11.891749,42810.0,...,2022-01-04,2022-01-09,5.0,0.01,0.656,1112.347052,0.000056,0.088656,2015,1
1,2015 AE45,(2015 AE45),2015-01-02 15:56:00,0.048239,7216474.0,18.77,0.048220,0.048258,7.065686,25436.0,...,2015-01-15,2015-02-19,35.0,0.10,0.827,30.863245,0.000957,0.218582,2015,1
2,613286,613286 (2005 YQ96),2015-01-02 21:46:00,0.026522,3967664.0,10.32,0.026522,0.026522,12.703378,45732.0,...,2005-12-30,2023-12-15,6559.0,17.96,0.495,52.080621,0.070619,3.375827,2015,1
3,2014 YQ34,(2014 YQ34),2015-01-03 13:29:00,0.079692,11921722.0,31.01,0.078275,0.081108,24.982094,89936.0,...,2014-12-23,2014-12-29,6.0,0.02,4.194,156.715248,0.002735,1.306441,2015,1
4,2014 YE42,(2014 YE42),2015-01-03 15:00:00,0.010995,1644896.0,4.28,0.010971,0.011020,13.998882,50396.0,...,2014-12-30,2015-01-04,5.0,0.01,2.839,698.812020,0.005507,1.038854,2015,1


In [4]:
print(df.columns.tolist())

['designation', 'full_name', 'close_approach_date', 'distance_au', 'dist_km', 'dist_lunar', 'distance_min_au', 'distance_max_au', 'velocity_km_s', 'v_rel_kmh', 'velocity_infinity_km_s', 'absolute_magnitude', 'is_future', 'spkid', 'pdes', 'pha', 'H', 'diameter_km', 'diameter_m', 'diameter_is_estimated', 'size_category', 'class', 'e', 'a', 'i', 'q', 'ad', 'per', 'per_y', 'moid_au', 'moid_km', 'moid_lunar_distances', 'n', 'condition_code', 'first_obs', 'last_obs', 'data_arc', 'data_arc_years', 'orbit_stretch', 'interaction_index', 'size_energy_index', 'diameter_velocity_proxy', 'year', 'month']


In [5]:
hazard_model = joblib.load(HAZARD_MODEL_PATH)

distance_model = joblib.load(DISTANCE_MODEL_PATH)

velocity_model = joblib.load(VELOCITY_MODEL_PATH)

In [6]:
hazard_features = list(hazard_model.feature_names_in_)

print(hazard_features)

['diameter_km', 'H', 'e', 'a', 'i', 'q', 'ad', 'per_y', 'moid_au', 'n', 'condition_code', 'orbit_stretch', 'interaction_index', 'size_energy_index', 'diameter_velocity_proxy']


In [7]:
future_df = df[
    df["is_future"] == True
].copy()

print("Future rows:", len(future_df))

Future rows: 3885


In [8]:
X_future = future_df[hazard_features].copy()

future_df["hazard_prob"] = (
    hazard_model.predict_proba(X_future)[:, 1]
)

future_df["distance_pred"] = (
    distance_model.predict(X_future)
)

future_df["velocity_pred"] = (
    velocity_model.predict(X_future)
)

In [9]:
future_df = future_df.sort_values(
    "hazard_prob",
    ascending=False
)

future_df = future_df.head(TOP_N_ASTEROIDS)

future_df.head()

,designation,full_name,close_approach_date,distance_au,dist_km,dist_lunar,distance_min_au,distance_max_au,velocity_km_s,v_rel_kmh,...,data_arc_years,orbit_stretch,interaction_index,size_energy_index,diameter_velocity_proxy,year,month,hazard_prob,distance_pred,velocity_pred
26363,613450,613450 (2006 PY17),2033-02-27 03:32:00,0.045043,6738308.0,17.53,0.045040,0.045046,16.212618,58365.0,...,13.77,2.428,38.608548,0.240393,7.949030,2033,2,0.999999,0.063020,17.094345
26764,2015 DC200,(2015 DC200),2034-03-20 04:31:00,0.042708,6389003.0,16.62,0.042704,0.042712,32.580753,117291.0,...,9.87,3.072,33.002211,0.068694,8.539296,2034,3,0.999999,0.056172,32.029633
26367,2013 GS66,(2013 GS66),2033-03-03 04:35:00,0.078261,11707715.0,30.46,0.078234,0.078289,10.905612,39260.0,...,2.64,2.354,46.080826,0.050225,2.444055,2033,3,0.999999,0.052765,12.802992
24593,612326,612326 (2002 CD14),2028-09-06 08:59:00,0.044975,6728094.0,17.50,0.044973,0.044976,16.873652,60745.0,...,19.75,2.063,34.841992,0.085688,4.939353,2028,9,0.999999,0.060320,15.299498
24240,2017 QL33,(2017 QL33),2027-11-02 16:17:00,0.020793,3110659.0,8.09,0.020793,0.020794,7.821474,28157.0,...,10.54,1.765,48.778108,0.036722,1.498820,2027,11,0.999999,0.046069,7.922069


In [10]:
possible_a = [
    "a",
    "semi_major_axis",
    "semi_major_axis_au"
]

possible_e = [
    "e",
    "eccentricity"
]

possible_i = [
    "i",
    "inclination"
]

a_col = next(c for c in possible_a if c in future_df.columns)

e_col = next(c for c in possible_e if c in future_df.columns)

i_col = next(c for c in possible_i if c in future_df.columns)

print(a_col, e_col, i_col)

a e i


In [11]:
def generate_orbit(
    a,
    e,
    inc_deg,
    points=720
):

    theta = np.linspace(
        0,
        2*np.pi,
        points
    )

    r = (
        a*(1-e**2)
        /
        (1+e*np.cos(theta))
    )

    x = r*np.cos(theta)

    y = r*np.sin(theta)

    inc = np.radians(inc_deg)

    z = y*np.sin(inc)

    y = y*np.cos(inc)

    return x,y,z

In [12]:
asteroids = []

In [13]:
for _, row in future_df.iterrows():

    a = float(row[a_col])

    e = float(row[e_col])

    inc = float(row[i_col])

    x,y,z = generate_orbit(
        a,
        e,
        inc,
        N_TRAJECTORY_POINTS
    )

    orbit = []

    for k in range(len(x)-1):

        vx = x[k+1]-x[k]

        vy = y[k+1]-y[k]

        vz = z[k+1]-z[k]

        speed = np.sqrt(
            vx*vx +
            vy*vy +
            vz*vz
        )

        orbit.append({

            "t": int(k),

            "position": {

                "x": float(x[k]),

                "y": float(y[k]),

                "z": float(z[k])

            },

            "velocity": {

                "vx": float(vx),

                "vy": float(vy),

                "vz": float(vz)

            },

            "direction": {

                "dx": float(vx/speed),

                "dy": float(vy/speed),

                "dz": float(vz/speed)

            }

        })

    asteroid_id = (
        str(row["spkid"])
        if "spkid" in row.index
        else str(_)
    )

    asteroids.append({

        "asteroid_id": asteroid_id,

        "hazard_probability":
        float(row["hazard_prob"]),

        "predicted_distance_au":
        float(row["distance_pred"]),

        "predicted_velocity":
        float(row["velocity_pred"]),

        "trajectory":
        orbit

    })

In [14]:
with open(
    OUTPUT_JSON,
    "w"
) as f:

    json.dump(
        {
            "asteroids": asteroids
        },
        f,
        indent=2
    )

print("Saved:", OUTPUT_JSON)

Saved: ../data/exports/asteroid_trajectories.json


In [15]:
with open(
    OUTPUT_JSON
) as f:

    sample = json.load(f)

print(
    sample["asteroids"][0].keys()
)

dict_keys(['asteroid_id', 'hazard_probability', 'predicted_distance_au', 'predicted_velocity', 'trajectory'])
